##### 코랩을 사용할 경우

In [1]:
try:
    # Google Drive를 Colab(/content/drive)에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec05
    %ls
except ImportError:
    pass

##### 이전 노트북 실행

In [2]:
%%capture
get_ipython().run_line_magic("run", "01_data_preprocessing.ipynb")

##### 임포트

In [3]:
%%capture
%pip install optuna
import optuna

import torch
import torch.nn as nn
import torch.optim as optim
import copy
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)  # 불필요한 로그 숨기기

##### Device 설정

In [4]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cpu


##### 표형식 데이터셋용 Optuna 하이퍼파라미터 설정

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# 하이퍼파라미터 탐색 설정: 개발자 직접 설정
# ══════════════════════════════════════════════════════════════════════════

# ── 재현성 설정 ────────────────────────────────────────────────────────────
USE_SEED = True   # True: 매 실행마다 동일한 탐색 결과 / False: 매 실행마다 다른 탐색 결과
SEED     = 42

# ── 모델 구조 ──────────────────────────────────────────────────────────────
IN_FEATURES = 13  # 입력 특징 수 (데이터셋에 맞게 설정)
OUT_CLASSES = 3   # 출력 클래스 수 (데이터셋에 맞게 설정)

# ── 하이퍼파라미터 탐색 범위 ────────────────────────────────────────────────
N_LAYERS_MIN = 1          # 은닉층 수 최솟값
N_LAYERS_MAX = 3          # 은닉층 수 최댓값
UNITS_MIN    = 16         # 은닉층 유닛 수 최솟값
UNITS_MAX    = 128        # 은닉층 유닛 수 최댓값
UNITS_STEP   = 16         # 은닉층 유닛 수 간격, 은닉층 유닛 수를 결정할때 테스트할 증감량
DROPOUT_MIN  = 0.0        # 드롭아웃 비율 최솟값
DROPOUT_MAX  = 0.5        # 드롭아웃 비율 최댓값
LR_MIN       = 1e-4       # 학습률 최솟값
LR_MAX       = 1e-1       # 학습률 최댓값
BATCH_SIZES  = [8, 16, 32, 64]             # 미니배치 크기 후보
OPTIMIZERS   = ["Adam", "SGD", "RMSprop"]  # 옵티마이저 후보

# ── 탐색 설정 ──────────────────────────────────────────────────────────────
# 얼마나 많은 하이퍼파라미터 조합을 시도할지
# 50이면 서로 다른 조합 50개를 탐색
# 클수록 더 좋은 조합을 찾을 가능성이 높지만, 시간이 그만큼 오래 걸림
N_TRIALS      = 50  # Optuna Trial 수

# 하나의 Trial(하이퍼파라미터 조합 1번 시도)에서 학습을 몇 에포크 돌릴지
# 탐색 중에는 빠른 평가가 목적이므로 짧게 설정
# 나중에 최적 조합이 정해진 뒤 최종 훈련에서는 더 많은 에포크(예: 100)로 돌림
SEARCH_EPOCHS = 50  # 탐색 시 에포크 수 (탐색 속도를 위해 짧게 설정)


In [6]:
# 모델 생성 함수
def build_model(trial):
    """
    Trial이 제안한 하이퍼파라미터로 모델을 동적으로 생성
    하이퍼파라미터: n_layers, n_units_l{i}, dropout
    1) 은닉층 수(n_layers)만큼 반복
    2) 각 은닉층의 유닛 수(n_units_l{i})를 Trial이 제안한 값으로 설정
    3) 각 은닉층 뒤에 ReLU 활성화 함수와 드롭아웃(dropout) 적용
    4) 마지막 출력층은 클래스 수(OUT_CLASSES)로 설정
    5) nn.Sequential로 모델 반환
    """
    n_layers = trial.suggest_int("n_layers", N_LAYERS_MIN, N_LAYERS_MAX)
    dropout  = trial.suggest_float("dropout", DROPOUT_MIN, DROPOUT_MAX)

    layers      = []
    in_features = IN_FEATURES
    for i in range(n_layers):
        out_features = trial.suggest_int(f"n_units_l{i}", UNITS_MIN, UNITS_MAX, step=UNITS_STEP)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        in_features = out_features
    layers.append(nn.Linear(in_features, OUT_CLASSES))

    return nn.Sequential(*layers)

# 목적 함수 정의: 1회 시도시 검증 손실을 최소화하는 하이퍼파라미터 탐색
def objective(trial):
    """Optuna가 최소화할 목적 함수 → 검증 손실 반환"""
    if USE_SEED:
        torch.manual_seed(SEED)  # 가중치 초기화 / 드롭아웃 랜덤성 고정

    lr             = trial.suggest_float("lr", LR_MIN, LR_MAX, log=True)
    batch_size     = trial.suggest_categorical("batch_size", BATCH_SIZES)
    optimizer_name = trial.suggest_categorical("optimizer", OPTIMIZERS)

    model     = build_model(trial).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    for epoch in range(SEARCH_EPOCHS):
        # ── 훈련 ──────────────────────────────────────────────────
        model.train()
        for i in range(0, len(train_x_tensor), batch_size):
            batch_x_tensor = train_x_tensor[i:i + batch_size]
            batch_y_tensor = train_y_tensor[i:i + batch_size]
            optimizer.zero_grad()
            model_outputs = model(batch_x_tensor)
            loss = loss_fn(model_outputs, batch_y_tensor)
            loss.backward()
            optimizer.step()

        # ── 검증 손실 계산 ─────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            model_outputs = model(val_x_tensor)
            val_loss = loss_fn(model_outputs, val_y_tensor).item()

        # 현재 epoch의 검증 손실을 Optuna에 보고하는 메서드
        trial.report(val_loss, epoch)
        # 가지치기(조기 종료) 필요 여부 확인
        if trial.should_prune():
            # 가지치기(조기 종료)하기: TrialPruned 예외를 발생시켜 조기 종료시킴
            raise optuna.exceptions.TrialPruned()

    return val_loss

# Optuna Study 생성: 하이퍼파라미터 탐색을 관리하는 객체
study = optuna.create_study(
    # 검증 손실 최소화가 목표이므로 "minimize"로 설정
    direction="minimize",
    # 중간값보다 나쁜 Trial은 가지치기(조기 종료)하여 탐색 속도 향상
    # 10 에포크 워밍업 후 가지치기 시작
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    # Study 이름
    study_name="wine-hyperparameter-search",
    # 샘플러 설정: 다음 Trial에서 어떤 하이퍼파라미터 조합을 시도할지 결정하는 전략
    # TPESampler: Optuna의 기본 샘플러로, 단순 무작위 탐색이 아니라 이전 Trial 결과를 학습해 더 좋은 조합을 우선 탐색
    sampler=optuna.samplers.TPESampler(seed=SEED) if USE_SEED else optuna.samplers.TPESampler(),
)

# Optuna 최적화 실행: objective() 함수를 N_TRIALS번 실행하여 최적 하이퍼파라미터 탐색
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# Optuna 최적화 결과 출력
print("\n" + "=" * 60)
print("Optuna 최적화 완료")
print("=" * 60)
print(f"  완료된 Trial 수  : {len(study.trials)}")
print(f"  최소 검증 손실   : {study.best_value:.4f}")
print(f"\n  최적 하이퍼파라미터:")
for k, v in study.best_params.items():
    print(f"    {k:20s}: {v}")


  0%|          | 0/50 [00:00<?, ?it/s]


Optuna 최적화 완료
  완료된 Trial 수  : 50
  최소 검증 손실   : 0.0000

  최적 하이퍼파라미터:
    lr                  : 0.07929051073698695
    batch_size          : 64
    optimizer           : Adam
    n_layers            : 2
    dropout             : 0.37657963847575704
    n_units_l0          : 48
    n_units_l1          : 64


##### 하이퍼파라미터 탐색 결과 시각화

In [7]:
from optuna.visualization import plot_optimization_history
plot_optimization_history(study)


스크린샷 그래프를 보면 빨간 선(Best Value)이 Trial 8~10 부근에서 거의 0에 수렴했고, <br>
이후 50번까지 더 탐색해도 빨간 선이 더 내려가지 않음 <br><br>

의미하는 바: <br>
---------------------------------------------------------------------------<br>
Trial ~8: 매우 좋은 하이퍼파라미터 조합 발견 <br>
Trial 8~50: 계속 탐색했지만 더 나은 조합을 못 찾음 <br>
따라서 이 데이터셋·탐색 범위에서는 사실상 N_TRIALS=8 정도로도 충분했을 가능성이 높음 <br>

실무에서는 이런 그래프를 보고 다음 번 실행 시 N_TRIALS를 줄이거나,  <br>
반대로 Best Value가 끝까지 계속 내려가고 있다면 N_TRIALS를 늘리는 판단 기준으로 활용 <br>

**그래프 용어 설명**

| 용어 | 설명 |
|------|------|
| **Trial** | Optuna가 하이퍼파라미터를 한 조합 선택해 모델을 학습·평가하는 1회 시도. `n_trials=50`이면 50번 반복 |
| **Objective Value** | 각 Trial에서 목적 함수가 반환한 값 (이 코드에서는 검증 손실). 점으로 표시됨 |
| **Best Value** | 현재까지 Trial 중 가장 낮은 Objective Value.

In [8]:
# 어떤 하이퍼파라미터가 검증 손실에 가장 큰 영향을 미쳤는지
from optuna.visualization import plot_param_importances
plot_param_importances(study)


##### 학습 / 검증 / 테스트

In [9]:
# 모데 구조 정의: 최적 하이퍼파라미터를 __init__ 인자로 받아 표형식 데이터 분류 모델 정의
# - n_units_l: 은닉층 유닛 수 목록, n_layers 수만큼 [n_units_l0 유닛수, n_units_l1 유닛수, ...]
# - dropout: 드롭아웃 비율
class TabularClassifier(nn.Module):
    def __init__(self, n_units_l: list, dropout: float):
        super().__init__()

        layers      = []
        in_features = IN_FEATURES

        for units in n_units_l:
            layers.append(nn.Linear(in_features, units))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_features = units

        layers.append(nn.Linear(in_features, OUT_CLASSES))  # 출력층
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# Optuna 최적 하이퍼파라미터 가져오기
best_params = study.best_params

# 모델 객체 생성
n_units_l = [ best_params[f"n_units_l{i}"] for i in range(best_params["n_layers"]) ]
optimized_model = TabularClassifier(n_units_l=n_units_l, dropout=best_params["dropout"]).to(device)

# 손실 함수 생성
optimized_loss_fn = nn.CrossEntropyLoss()

# 학습률
# optimized_lr = best_params["lr"]  # 검증 손실이 매우 불안정(너무 출렁임)
optimized_lr = 0.01

# 옵티마이저 생성
optimized_optimizer = getattr(optim, best_params["optimizer"])(
    optimized_model.parameters(), lr=optimized_lr
)

# 배치 사이즈 얻기
optimized_batch_size = best_params["batch_size"]


In [10]:
# 최적 하이퍼파라미터로 100 에폭 훈련
epochs = 100

# 조기 종료(Early Stopping) 설정 #####################################
# patience: 검증 손실이 개선되지 않아도 허용할 에포크 수
# patience_counter: 개선 없이 누적된 에포크 수
# best_val_loss: 현재까지 가장 낮은 검증 손실 (초기값: 무한대)
# best_model_state: 검증 손실이 가장 낮았을 때의 모델 파라미터 복사본
patience         = 10
patience_counter = 0
best_val_loss    = float('inf') # 양의 무한대로 초기화
best_model_state = None
####################################################################

for epoch in range(epochs):

    # 훈련하기 -----------------------------------------------------------
    optimized_model.train()
    train_loss    = 0
    train_correct = 0

    for i in range(0, len(train_x_tensor), optimized_batch_size):
        batch_x_tensor = train_x_tensor[i:i + optimized_batch_size]
        batch_y_tensor = train_y_tensor[i:i + optimized_batch_size]

        optimized_optimizer.zero_grad()
        preds = optimized_model(batch_x_tensor)
        loss  = optimized_loss_fn(preds, batch_y_tensor)
        loss.backward()
        optimized_optimizer.step()

        train_loss    += loss.item()
        train_correct += (preds.argmax(dim=1) == batch_y_tensor).sum().item()

    avg_train_loss = train_loss / (len(train_x_tensor) / optimized_batch_size)
    train_acc      = train_correct / len(train_x_tensor) * 100

    # 검증하기 -----------------------------------------------------------
    optimized_model.eval()
    with torch.no_grad():
        val_out     = optimized_model(val_x_tensor)
        val_loss    = optimized_loss_fn(val_out, val_y_tensor).item()
        val_correct = (val_out.argmax(dim=1) == val_y_tensor).sum().item()
    val_acc = val_correct / len(val_x_tensor) * 100

    # 조기 종료 #####################################
    if val_loss < best_val_loss:
        # 검증 손실이 개선된 경우: 최적 상태 갱신
        best_val_loss    = val_loss
        patience_counter = 0
        # 현재 에포크의 모델 파라미터를 별도 메모리에 복사
        best_model_state = copy.deepcopy(optimized_model.state_dict())
        # 출력하기
        print(f"Epoch [{epoch+1:3d}/{epochs}]  "
            f"훈련 손실: {avg_train_loss:.4f}  훈련 정확도: {train_acc:5.1f}%  "
            f"검증 손실: {val_loss:.4f}  검증 정확도: {val_acc:5.1f}%  최적 모델 저장")
    else:
        # 검증 손실이 개선되지 않은 경우: 카운터 증가
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n조기 종료: {patience} 에포크 동안 검증 손실 개선 없음")
            break
    # ##############################################

# 가장 성능이 좋았던 에포크의 가중치로 모델 복원
optimized_model.load_state_dict(best_model_state)
print(f"학습 완료  (최적 검증 손실: {best_val_loss:.4f})")

Epoch [  1/100]  훈련 손실: 1.3334  훈련 정확도:  52.8%  검증 손실: 0.7498  검증 정확도:  77.8%  최적 모델 저장
Epoch [  2/100]  훈련 손실: 0.8519  훈련 정확도:  90.1%  검증 손실: 0.4280  검증 정확도:  83.3%  최적 모델 저장
Epoch [  3/100]  훈련 손실: 0.4370  훈련 정확도:  92.3%  검증 손실: 0.2183  검증 정확도:  88.9%  최적 모델 저장
Epoch [  4/100]  훈련 손실: 0.2385  훈련 정확도:  95.8%  검증 손실: 0.0968  검증 정확도:  94.4%  최적 모델 저장
Epoch [  5/100]  훈련 손실: 0.2282  훈련 정확도:  95.1%  검증 손실: 0.0713  검증 정확도:  94.4%  최적 모델 저장
Epoch [  7/100]  훈련 손실: 0.1081  훈련 정확도:  99.3%  검증 손실: 0.0707  검증 정확도:  94.4%  최적 모델 저장
Epoch [  8/100]  훈련 손실: 0.0837  훈련 정확도:  97.9%  검증 손실: 0.0523  검증 정확도:  94.4%  최적 모델 저장

조기 종료: 10 에포크 동안 검증 손실 개선 없음
학습 완료  (최적 검증 손실: 0.0523)


In [11]:
# 테스트하기
optimized_model.eval()
with torch.no_grad():
    test_out      = optimized_model(test_x_tensor)
    test_loss     = optimized_loss_fn(test_out, test_y_tensor).item()
    test_preds    = test_out.argmax(dim=1)
    test_accuracy = (test_preds == test_y_tensor).float().mean().item() * 100

print(f"테스트 손실: {test_loss:.4f}  테스트 정확도: {test_accuracy:.1f}%  "
      f"({int(test_accuracy / 100 * len(test_x_tensor))}/{len(test_x_tensor)}개 정답)")


테스트 손실: 0.0015  테스트 정확도: 100.0%  (18/18개 정답)


In [12]:
save_dir = "saved_models_optuna"
os.makedirs(save_dir, exist_ok=True)

# 모델 구조와 파라미터를 함께 저장
full_path = os.path.join(save_dir, 'wine_classifier_full.pt')
torch.save(optimized_model, full_path)
print(f"✓ 전체 모델 저장 완료: {full_path}")


✓ 전체 모델 저장 완료: saved_models_optuna/wine_classifier_full.pt


In [13]:
# full.pt 파일을 이용
pt_path = os.path.join('saved_models_optuna', 'wine_classifier_full.pt')
model = torch.load(pt_path, map_location=device, weights_only=False)

# 예측해보기
model.eval()
with torch.inference_mode():  # torch.no_grad():
    test_outputs  = model(test_x_tensor[0:1])
    class_label = test_outputs.argmax(dim=1)
    print(f"예측된 클래스 레이블: {class_label.item()}")

예측된 클래스 레이블: 0
